In [8]:
import json
import pickle
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"

FILE_MAP = {
    "labels": "labelset.txt",
    "train_dataset": "train_dataset.jsonl"
}

with open(DATA_DIR / FILE_MAP["labels"], 'r', encoding='utf-8') as f:
    labelset = [line.strip() for line in f]

with open(DATA_DIR / FILE_MAP["train_dataset"], 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

In [9]:
import pandas as pd

labelset = pd.DataFrame(labelset)
labelset.columns = ['codes']
labelset[['category', 'code']] = labelset['codes'].str.extract(r'([A-Z])([0-9]*)')

print(f"### LABELSET SUMMARY ###")
print(f"The labelset contains {len(labelset)} distinct ICD-10 codes.")
print(f"These codes are distributed across medical categories (ICD-10 Chapters).")
print(f"\nBelow is the breakdown of how many distinct codes fall into each category:")
count_df = labelset.groupby('category').size().reset_index(name='count')
print(count_df)

### LABELSET SUMMARY ###
The labelset contains 324 distinct ICD-10 codes.
These codes are distributed across medical categories (ICD-10 Chapters).

Below is the breakdown of how many distinct codes fall into each category:
   category  count
0         A     14
1         B      5
2         C     88
3         D     17
4         E     17
5         F      4
6         G      4
7         I     62
8         J     19
9         K      7
10        L      4
11        M     12
12        N      7
13        O      6
14        P      4
15        Q      7
16        R     24
17        S      1
18        T     10
19        Y      2
20        Z     10


In [12]:
import pandas as pd
result_df = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']

    for annotation in entry['annotations']:
        result_df.append({
            'patient_id': patient_id,
            'start': annotation['start'],
            'end': annotation['end'],
            'code': annotation['code'],
            'mention': annotation['mention']
        })
result_df = pd.DataFrame(result_df)
result_df[['category', 'subcode']] = result_df['code'].str.extract(r'([A-Z])([0-9]*)')
unique_codes = result_df['code'].unique()
unique_codes_count = result_df['code'].drop_duplicates().shape[0]

print("### TRAINING SET SUMMARY (ICD-10 CHAPTERS) ###")
print(f"The training set contains {unique_codes_count} unique ICD-10 codes.")
print("\nThe counts show how many times a disease from that chapter was tagged in the training dataset.\n")
count_df = result_df.groupby('category').size().reset_index(name='count')
print(count_df)


### TRAINING SET SUMMARY (ICD-10 CHAPTERS) ###
The training set contains 144 unique ICD-10 codes.

The counts show how many times a disease from that chapter was tagged in the training dataset.

   category  count
0         A      5
1         C     40
2         D     65
3         E    616
4         F      1
5         G      7
6         I   5142
7         J    454
8         K     14
9         M     25
10        N    209
11        Q      6
12        R   1895
13        T      7
14        Y    412
15        Z   1270
